# Notebook 5: Evaluating Generated Materials -- MLIPs

**APS Tutorial T4 -- Generative AI for Physics: From Models to Materials**

In this notebook you will learn to:
- Use **CHGNet** (a universal machine-learning interatomic potential) for fast property prediction
- **Relax** crystal structures to find local energy minima
- Evaluate **thermodynamic stability** via the convex hull (energy above hull)
- **Screen SCIGEN-generated materials** across structural constraint types
- Understand the full **Generate -> Screen -> Validate** pipeline

> **Prerequisites:** Run `00_setup.ipynb` first. Notebook 04 is recommended but not required.

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Clone repo if needed
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install all tutorial dependencies from the shared requirements file
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 5.1 The evaluation problem

Generating crystal structures is only half the job. After generation, we need to ask:

1. **Is this structure physically reasonable?** (sensible bond lengths, coordination)
2. **Is it thermodynamically stable?** (low energy, near the convex hull)
3. **Does it have interesting properties?** (magnetic, electronic, mechanical)

### The traditional approach: DFT
Density Functional Theory (DFT) gives accurate energies and forces, but it's
**expensive** — each structure takes minutes to hours on a supercomputer.
If you generated 10,000 candidates, DFT relaxation could take weeks.

### The modern approach: MLIPs
Machine Learning Interatomic Potentials are trained on DFT data and can predict
energies/forces/stresses in **seconds** — 1000x faster than DFT.
They enable rapid screening of large candidate sets.

---
## 5.2 CHGNet: a universal MLIP

[CHGNet](https://github.com/CederGroupHub/chgnet) (Crystal Hamiltonian Graph Neural Network)
is a universal potential trained on the Materials Project database.
It can predict energies, forces, stresses, and magnetic moments for any inorganic material.

Let's load the pretrained model.

In [ ]:
from chgnet.model.model import CHGNet

chgnet = CHGNet.load()
print(f'CHGNet loaded: {sum(p.numel() for p in chgnet.parameters()):,} parameters')

---
## 5.3 Predicting properties

Let's predict the energy of a known structure from the training data.

In [ ]:
import os
import pandas as pd
from pymatgen.core import Structure

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'test.csv'))

# Pick a structure
row = df.iloc[0]
struct = Structure.from_str(row['cif'], fmt='cif')

print(f'Structure: {row["pretty_formula"]} ({row["material_id"]})')
print(f'Atoms: {struct.num_sites}')
print(f'DFT formation energy: {row["formation_energy_per_atom"]:.4f} eV/atom')

In [ ]:
# Predict with CHGNet
prediction = chgnet.predict_structure(struct)

print(f'CHGNet predictions for {row["pretty_formula"]}:')
print(f'  Energy:  {prediction["e"]:>10.4f} eV/atom')
print(f'  Forces:  max |F| = {abs(prediction["f"]).max():.4f} eV/Å')
print(f'  Stress:  {prediction["s"].diagonal().tolist()}')
if 'm' in prediction:
    print(f'  Magmom:  {prediction["m"].tolist()}')

---
## 5.4 Structure relaxation from a perturbed state

To demonstrate relaxation clearly, we start from a **deliberately perturbed**
structure — we take a known stable crystal and displace atoms randomly by ~0.3 Å.
This simulates what a generated (unrelaxed) structure might look like.

Then we relax with CHGNet and watch:
1. The **energy decrease** step by step toward equilibrium
2. The **atoms settle** back to their stable positions

In [ ]:
import numpy as np
from chgnet.model.dynamics import StructOptimizer

# --- Create a perturbed (unstable) structure ---
np.random.seed(42)
perturbed = struct.copy()

# Add random displacements of ~0.3 Å to each atom
noise_amplitude = 0.3  # Å
cart_coords = perturbed.cart_coords.copy()
noise = noise_amplitude * np.random.randn(*cart_coords.shape)
for i, site in enumerate(perturbed):
    perturbed.translate_sites(i, noise[i], frac_coords=False)

# Check: energy of perturbed structure is HIGHER than the original
pred_orig = chgnet.predict_structure(struct)
pred_pert = chgnet.predict_structure(perturbed)
print(f'Original (equilibrium):  E = {pred_orig["e"]:.4f} eV/atom,  max|F| = {abs(pred_orig["f"]).max():.4f} eV/Å')
print(f'Perturbed (noisy):       E = {pred_pert["e"]:.4f} eV/atom,  max|F| = {abs(pred_pert["f"]).max():.2f} eV/Å')
print(f'Energy increase from perturbation: {pred_pert["e"] - pred_orig["e"]:.4f} eV/atom')
print(f'\nRelaxing perturbed structure with CHGNet...')

# --- Relax the perturbed structure ---
optimizer = StructOptimizer()
result = optimizer.relax(perturbed, verbose=True)

relaxed = result['final_structure']
e_final = result['trajectory'].energies[-1] / relaxed.num_sites
print(f'\nRelaxation complete:')
print(f'  Perturbed energy: {pred_pert["e"]:.4f} eV/atom')
print(f'  Relaxed energy:   {e_final:.4f} eV/atom')
print(f'  DFT reference:    {row["formation_energy_per_atom"]:.4f} eV/atom (formation energy)')
print(f'  Energy drop:      {pred_pert["e"] - e_final:.4f} eV/atom')

In [ ]:
# Compare: how close did relaxation get to the original equilibrium structure?
print('=== Comparison: perturbed → relaxed vs. original equilibrium ===\n')

print('Lattice parameter changes (relaxed vs. original):')
for param in ['a', 'b', 'c', 'alpha', 'beta', 'gamma']:
    orig = getattr(struct.lattice, param)
    relax = getattr(relaxed.lattice, param)
    unit = 'Å' if param in ['a', 'b', 'c'] else '°'
    print(f'  {param:>5s}: orig={orig:8.3f}  relaxed={relax:8.3f} {unit}  (Δ = {relax - orig:+.4f})')

# Displacement: perturbed → relaxed (how far did atoms move during relaxation?)
disp_relax = np.linalg.norm(relaxed.cart_coords - perturbed.cart_coords, axis=1)
# Displacement: relaxed → original (how close to equilibrium did we get?)
disp_equil = np.linalg.norm(relaxed.cart_coords - struct.cart_coords, axis=1)
# Original perturbation magnitude
disp_noise = np.linalg.norm(perturbed.cart_coords - struct.cart_coords, axis=1)

species_list = [str(s.specie) for s in struct]
print(f'\nPer-atom displacements:')
print(f'  {"Atom":<6} {"Perturbation":<16} {"Relaxation move":<18} {"Residual from eq.":<18}')
for i in range(struct.num_sites):
    print(f'  {species_list[i]:<6} {disp_noise[i]:>10.4f} Å      {disp_relax[i]:>10.4f} Å        {disp_equil[i]:>10.4f} Å')

print(f'\nMean perturbation:      {disp_noise.mean():.4f} Å')
print(f'Mean residual from eq.: {disp_equil.mean():.4f} Å')
print(f'Recovery ratio:         {(1 - disp_equil.mean() / disp_noise.mean()) * 100:.1f}%')

### Relaxation trajectory

Let's visualize how the energy decreased during relaxation:

In [ ]:
import matplotlib.pyplot as plt

# Plot energy trajectory during relaxation
energies_traj = result['trajectory'].energies
n_atoms_relax = relaxed.num_sites
e_per_atom = [e / n_atoms_relax for e in energies_traj]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Energy trajectory
ax = axes[0]
steps = range(len(e_per_atom))
ax.plot(steps, e_per_atom, 'o-', color='steelblue', markersize=5, linewidth=1.5,
        label='CHGNet relaxation')
ax.axhline(pred_orig['e'], color='green', linestyle='--', alpha=0.7,
           label=f'Original equilibrium ({pred_orig["e"]:.4f} eV/atom)')
ax.axhline(pred_pert['e'], color='red', linestyle=':', alpha=0.7,
           label=f'Perturbed start ({pred_pert["e"]:.4f} eV/atom)')
ax.set_xlabel('Optimization step', fontsize=12)
ax.set_ylabel('Energy (eV/atom)', fontsize=12)
ax.set_title('Relaxation trajectory: energy minimization', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: Max force trajectory (if available)
ax2 = axes[1]
try:
    forces_traj = result['trajectory'].forces
    max_forces = [np.max(np.linalg.norm(f, axis=1)) for f in forces_traj]
    ax2.semilogy(range(len(max_forces)), max_forces, 'o-', color='coral',
                 markersize=5, linewidth=1.5)
    ax2.axhline(0.05, color='gray', linestyle='--', alpha=0.5,
                label='Convergence threshold (0.05 eV/Å)')
    ax2.set_xlabel('Optimization step', fontsize=12)
    ax2.set_ylabel('Max |Force| (eV/Å)', fontsize=12)
    ax2.set_title('Force convergence', fontsize=13)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
except Exception:
    ax2.text(0.5, 0.5, 'Force trajectory not available', ha='center', va='center',
             transform=ax2.transAxes, fontsize=12)

plt.tight_layout()
plt.show()

print(f'Converged in {len(energies_traj)} steps')
print(f'Energy drop: {e_per_atom[0] - e_per_atom[-1]:.4f} eV/atom')

### Structure animation during relaxation

The animation below shows three things simultaneously:
1. **Colored spheres** — current atomic positions at each optimization step
2. **Red arrows** — forces acting on each atom (direction the optimizer pushes them)
3. **Small green dots** — equilibrium positions (from the original unperturbed structure)

Watch how the atoms (spheres) migrate toward the equilibrium positions (green dots)
while the force arrows shrink to zero.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np

# Extract trajectory frames
traj = result['trajectory']
n_frames = len(traj.energies)

# Select key frames (evenly spaced, max 20 frames for smooth animation)
n_show = min(n_frames, 20)
frame_indices = np.linspace(0, n_frames - 1, n_show, dtype=int)

# Equilibrium positions (original unperturbed structure)
equil_pos = struct.cart_coords

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')

# Get coordinate range for consistent axis limits
all_positions = np.array([traj.atom_positions[i] for i in frame_indices])
pad = 0.5
x_lim = (all_positions[:, :, 0].min() - pad, all_positions[:, :, 0].max() + pad)
y_lim = (all_positions[:, :, 1].min() - pad, all_positions[:, :, 1].max() + pad)
z_lim = (all_positions[:, :, 2].min() - pad, all_positions[:, :, 2].max() + pad)

# Force scale: normalize arrows for visibility
has_forces = hasattr(traj, 'forces') and traj.forces is not None and len(traj.forces) > 0
if has_forces:
    all_forces = np.array([traj.forces[i] for i in frame_indices])
    max_force_global = np.max(np.linalg.norm(all_forces, axis=2))
    arrow_scale = 1.0 / max(max_force_global, 1e-6)  # normalize to ~1 Å length

species_list = [str(s.specie) for s in struct]
unique_el = sorted(set(species_list))
cmap_colors = plt.cm.tab10(np.linspace(0, 0.9, max(len(unique_el), 1)))
color_map = {el: cmap_colors[i] for i, el in enumerate(unique_el)}

def update(frame_num):
    ax.clear()
    idx = frame_indices[frame_num]
    positions = traj.atom_positions[idx]
    energy = traj.energies[idx] / struct.num_sites

    # Draw equilibrium positions (small green dots)
    ax.scatter(equil_pos[:, 0], equil_pos[:, 1], equil_pos[:, 2],
               s=30, c='limegreen', alpha=0.6, marker='o',
               edgecolors='green', linewidths=0.5, label='Equilibrium')

    # Draw current atom positions
    for el in unique_el:
        mask = [s == el for s in species_list]
        coords = positions[mask]
        ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                   s=200, c=[color_map[el]], label=el,
                   alpha=0.85, edgecolors='k', linewidths=0.5)

    # Draw force arrows
    if has_forces and idx < len(traj.forces):
        forces = traj.forces[idx]
        for i in range(len(positions)):
            f = forces[i]
            f_mag = np.linalg.norm(f)
            if f_mag > 0.01:  # skip tiny forces
                ax.quiver(positions[i, 0], positions[i, 1], positions[i, 2],
                          f[0] * arrow_scale, f[1] * arrow_scale, f[2] * arrow_scale,
                          color='red', alpha=0.7, linewidth=1.5,
                          arrow_length_ratio=0.3)

    ax.set_xlim(*x_lim); ax.set_ylim(*y_lim); ax.set_zlim(*z_lim)
    ax.set_xlabel('x (Å)'); ax.set_ylabel('y (Å)'); ax.set_zlabel('z (Å)')

    # Build legend without duplicates
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=7)

    max_f = np.max(np.linalg.norm(traj.forces[idx], axis=1)) if has_forces and idx < len(traj.forces) else 0
    ax.set_title(f'Step {idx}/{n_frames-1}  |  E = {energy:.4f} eV/atom  |  max|F| = {max_f:.3f} eV/Å',
                 fontsize=10)

anim = FuncAnimation(fig, update, frames=n_show, interval=400, repeat=True)
plt.close(fig)

try:
    display(HTML(anim.to_jshtml()))
except Exception:
    # Fallback: show first, middle, and last frames as static plots
    print('Animation display not supported — showing key frames instead.')
    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5), subplot_kw={'projection': '3d'})
    for ax2, label, fi in zip(axes2, ['Initial (perturbed)', 'Mid relaxation', 'Final (relaxed)'],
                               [0, n_show // 2, n_show - 1]):
        idx = frame_indices[fi]
        positions = traj.atom_positions[idx]
        energy = traj.energies[idx] / struct.num_sites
        # Equilibrium
        ax2.scatter(equil_pos[:, 0], equil_pos[:, 1], equil_pos[:, 2],
                    s=30, c='limegreen', alpha=0.6, edgecolors='green', linewidths=0.5)
        # Current positions
        for el in unique_el:
            mask = [s == el for s in species_list]
            coords = positions[mask]
            ax2.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                       s=200, c=[color_map[el]], label=el,
                       alpha=0.85, edgecolors='k', linewidths=0.5)
        # Forces
        if has_forces and idx < len(traj.forces):
            forces = traj.forces[idx]
            for i in range(len(positions)):
                f = forces[i]
                if np.linalg.norm(f) > 0.01:
                    ax2.quiver(positions[i, 0], positions[i, 1], positions[i, 2],
                              f[0]*arrow_scale, f[1]*arrow_scale, f[2]*arrow_scale,
                              color='red', alpha=0.7, linewidth=1.5, arrow_length_ratio=0.3)
        ax2.set_xlim(*x_lim); ax2.set_ylim(*y_lim); ax2.set_zlim(*z_lim)
        ax2.set_xlabel('x (Å)'); ax2.set_ylabel('y (Å)'); ax2.set_zlabel('z (Å)')
        ax2.set_title(f'{label}\n(step {idx}, E={energy:.4f})', fontsize=10)
    plt.tight_layout()
    plt.show()

print('Green dots = equilibrium positions | Red arrows = forces | Spheres = current positions')

---
## 5.5 Batch evaluation

One of the key advantages of MLIPs is speed. Let's evaluate many structures
and compare CHGNet formation energies against DFT values.

In [ ]:
import matplotlib.pyplot as plt
import time
import numpy as np

# Evaluate the first 20 structures in the test set
n_eval = min(20, len(df))
energies_chgnet = []
energies_dft = []
formulas = []

# Compute CHGNet elemental reference energies for formation energy calculation
# (energy of each element in its standard bulk form)
from pymatgen.core import Structure, Lattice, Element

def get_elemental_ref_energies(chgnet_model):
    """Get CHGNet total energy per atom for common elemental reference structures."""
    refs = {}
    # Use simple structures as references
    # In practice, the MP elemental references are more accurate,
    # but this gives a reasonable approximation for comparison
    for el_symbol in ['Li', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ti', 'V', 'Cr',
                      'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se',
                      'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Ru', 'Rh', 'Pd', 'Ag',
                      'Cd', 'In', 'Sn', 'Sb', 'Te', 'Cs', 'Ba', 'La', 'Hf', 'Ta',
                      'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Pb', 'Bi',
                      'O', 'S', 'N', 'F', 'Cl', 'Br', 'I', 'P', 'C', 'B', 'H']:
        try:
            el = Element(el_symbol)
            # BCC structure as rough reference
            lat = Lattice.cubic(3.0)
            s = Structure(lat, [el_symbol, el_symbol], [[0,0,0], [0.5,0.5,0.5]])
            pred = chgnet_model.predict_structure(s)
            refs[el_symbol] = float(pred['e'])
        except:
            pass
    return refs

print('Computing elemental reference energies...')
elem_refs = get_elemental_ref_energies(chgnet)
print(f'  References computed for {len(elem_refs)} elements')

start = time.time()
for i in range(n_eval):
    row = df.iloc[i]
    try:
        s = Structure.from_str(row['cif'], fmt='cif')
        pred = chgnet.predict_structure(s)

        # Compute formation energy: E_total - sum(x_i * E_ref_i)
        total_e = float(pred['e'])  # eV/atom
        comp = s.composition.fractional_composition
        ref_sum = sum(comp[el] * elem_refs.get(str(el), 0) for el in comp)
        form_e = total_e - ref_sum

        energies_chgnet.append(form_e)
        energies_dft.append(row['formation_energy_per_atom'])
        formulas.append(row['pretty_formula'])
    except Exception as e:
        print(f'  Skipping {row["pretty_formula"]}: {e}')

elapsed = time.time() - start
print(f'Evaluated {len(energies_chgnet)} structures in {elapsed:.1f}s')
print(f'({elapsed/len(energies_chgnet)*1000:.0f} ms per structure)')

### Speed comparison: DFT vs. MLIP

One of the key advantages of MLIPs — they're orders of magnitude faster than DFT:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

methods = ['DFT\n(VASP/QE)', 'CHGNet\n(MLIP)']
times = [3600, elapsed / len(energies_chgnet)]  # seconds per structure
colors = ['#E69F00', 'steelblue']

bars = ax.barh(methods, times, color=colors, edgecolor='white', height=0.5)
ax.set_xscale('log')
ax.set_xlabel('Time per structure (seconds)', fontsize=11)
ax.set_title('DFT vs. MLIP evaluation speed', fontsize=13)

# Annotate bars
for bar, t in zip(bars, times):
    if t >= 60:
        label = f'{t/60:.0f} min'
    else:
        label = f'{t:.2f} s'
    ax.text(bar.get_width() * 1.5, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0.001, 50000)
plt.tight_layout()
plt.show()

speedup = 3600 / (elapsed / len(energies_chgnet))
print(f'CHGNet is ~{speedup:.0f}x faster than DFT for energy evaluation')

In [ ]:
if not energies_dft:
    print('No data — run the batch evaluation cell above first.')
else:
    fig, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(energies_dft, energies_chgnet, s=50, c='steelblue',
               edgecolors='white', linewidths=0.5)

    # Diagonal line
    all_vals = energies_dft + energies_chgnet
    lims = [min(all_vals) - 0.2, max(all_vals) + 0.2]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='Perfect agreement')

    # Compute MAE
    mae = np.mean(np.abs(np.array(energies_dft) - np.array(energies_chgnet)))

    ax.set_xlabel('DFT formation energy (eV/atom)', fontsize=12)
    ax.set_ylabel('CHGNet formation energy (eV/atom)', fontsize=12)
    ax.set_title(f'CHGNet vs. DFT — Formation Energy\nMAE = {mae:.3f} eV/atom', fontsize=13)
    ax.legend()
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

    print(f'Note: CHGNet elemental references are approximate (BCC structures).')
    print(f'For production use, use MP-computed elemental reference energies.')

---
## 5.6 Phonon calculation with CHGNet

Phonons (lattice vibrations) are a fundamental test of structural stability:
- **No imaginary frequencies** → structure is at a true energy minimum (dynamically stable)
- **Imaginary frequencies** → structure is a saddle point (dynamically unstable)

We use the [ASE Phonons](https://wiki.fysik.dtu.dk/ase/ase/phonons.html) module
with CHGNet as the force calculator. This computes the **dynamical matrix** via
finite displacements and extracts the phonon density of states (DOS).

> **Note:** For production phonon calculations, use [Phonopy](https://phonopy.github.io/phonopy/)
> with larger supercells. The ASE approach here is a quick tutorial demonstration.

---
## 5.7 Thermodynamic stability: energy above the convex hull

The **convex hull** of formation energies defines the set of thermodynamically stable
phases. A structure's **energy above hull** ($E_\text{hull}$) measures how far it is
from the nearest stable decomposition:

- $E_\text{hull} = 0$ → on the hull (thermodynamically stable)
- $E_\text{hull} < 0.025$ eV/atom → likely synthesizable (metastable)
- $E_\text{hull} > 0.1$ eV/atom → likely unstable (will decompose)

We use pymatgen's `PhaseDiagram` with the test set entries as reference phases.
This is an approximation — for production use, load the full Materials Project convex hull.

> **Note:** Computing E_hull against the full MP database requires an API key
> (`mp-api`). Here we build a local convex hull from the test set itself as a
> demonstration of the concept.

In [ ]:
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry
from pymatgen.core import Composition

# Build a phase diagram from the test set (using DFT formation energies as ground truth)
# In production, you'd use the full Materials Project database
n_hull = min(200, len(df))
pd_entries = []

for i in range(n_hull):
    r = df.iloc[i]
    try:
        comp = Composition(r['pretty_formula'])
        # PDEntry wants total energy per atom (not formation energy)
        # formation_energy_per_atom is relative to elemental references
        pd_entries.append(PDEntry(comp, r['formation_energy_per_atom'],
                                  name=r.get('material_id', f'mp-{i}')))
    except Exception:
        pass

print(f'Built phase diagram from {len(pd_entries)} entries')

try:
    phase_diag = PhaseDiagram(pd_entries)
    print(f'Stable phases: {len(phase_diag.stable_entries)}')
    print(f'Elements in hull: {[str(e) for e in phase_diag.elements]}')

    # Compute E_hull for the first few structures
    print(f'\n{"#":>3} {"Formula":<20} {"E_form (DFT)":<14} {"E_hull":<10} {"Status"}')
    print('-' * 65)

    e_hulls_dft = []
    for i in range(min(20, len(pd_entries))):
        entry = pd_entries[i]
        try:
            e_hull = phase_diag.get_e_above_hull(entry)
            e_hulls_dft.append(e_hull)
            status = 'STABLE' if e_hull < 1e-4 else ('metastable' if e_hull < 0.025 else 'unstable')
            print(f'{i:3d} {entry.name:<20} {entry.energy_per_atom:>10.4f}    {e_hull:>8.4f}  {status}')
        except Exception:
            e_hulls_dft.append(None)

    # Plot E_hull distribution
    valid_hulls = [h for h in e_hulls_dft if h is not None]
    if valid_hulls:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(valid_hulls, bins=20, color='steelblue', edgecolor='white')
        ax.axvline(0, color='green', linewidth=2, label='Convex hull (stable)')
        ax.axvline(0.025, color='orange', linestyle='--', label='Metastability threshold')
        ax.axvline(0.1, color='red', linestyle='--', label='Instability threshold')
        ax.set_xlabel('Energy above hull (eV/atom)', fontsize=12)
        ax.set_ylabel('Count', fontsize=12)
        ax.set_title('Thermodynamic stability distribution', fontsize=13)
        ax.legend(fontsize=9)
        ax.set_xlim(-0.01, max(valid_hulls) + 0.02)
        plt.tight_layout()
        plt.show()

except Exception as e:
    print(f'Phase diagram construction failed: {e}')
    print('This can happen if the test set lacks elemental reference entries.')
    print('For full E_hull analysis, use mp-api to load the Materials Project hull.')

---
## 5.8 Screening SCIGEN-generated materials with CHGNet

So far we have used CHGNet on **known** structures from the MP-20 test set.
Now let's apply it to **SCIGEN-generated** materials -- the real use case.

We will:
1. Download 24,743 DFT-validated SCIGEN structures from Figshare
2. Sample a subset across constraint types (kagome, honeycomb, triangular, ...)
3. Run CHGNet energy/force predictions
4. Relax selected structures and measure displacement
5. Assess which generated materials look thermodynamically stable

This demonstrates **MLIP as a fast post-generation screening tool** --
evaluating thousands of candidates in minutes instead of months of DFT.

In [ ]:
# === Download SCIGEN DFT-validated structures (24,743 materials) ===
import os, zipfile
from urllib.request import urlretrieve

SCIGEN_DFT_URL = 'https://api.figshare.com/v2/file/download/57245942'
SCIGEN_DFT_DIR = os.path.join(PROJECT_DIR, 'data', 'scigen_generated', '03_scigen_materials_dft')

# Check both possible directory names (NB04 may have used a different name)
alt_dir = os.path.join(PROJECT_DIR, 'data', 'scigen_generated', '03_scigen_materials_relaxed')
if os.path.exists(alt_dir) and not os.path.exists(SCIGEN_DFT_DIR):
    SCIGEN_DFT_DIR = alt_dir

if not os.path.exists(SCIGEN_DFT_DIR) or len(os.listdir(SCIGEN_DFT_DIR)) < 100:
    zip_path = os.path.join(PROJECT_DIR, 'data', 'scigen_generated', 'scigen_dft.zip')
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)

    if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 1000:
        print('Downloading SCIGEN DFT structures (~28 MB)...')
        import time
        for attempt in range(5):
            try:
                urlretrieve(SCIGEN_DFT_URL, zip_path)
                if os.path.getsize(zip_path) > 1000:
                    print(f'Downloaded: {os.path.getsize(zip_path)/1e6:.1f} MB')
                    break
                time.sleep(3)
            except Exception as e:
                print(f'Attempt {attempt+1}: {e}')
                time.sleep(3)

    if os.path.exists(zip_path) and os.path.getsize(zip_path) > 1000:
        print('Extracting...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(os.path.dirname(SCIGEN_DFT_DIR))
        # Handle possible directory name mismatch
        for name in ['03_scigen_materials_dft', '03_scigen_materials_relaxed']:
            candidate = os.path.join(PROJECT_DIR, 'data', 'scigen_generated', name)
            if os.path.exists(candidate):
                SCIGEN_DFT_DIR = candidate
                break
        print(f'Extracted to {SCIGEN_DFT_DIR}')
    else:
        print('Download failed. Try running this cell again.')

if os.path.exists(SCIGEN_DFT_DIR):
    from collections import Counter
    all_dirs = [d for d in os.listdir(SCIGEN_DFT_DIR)
                if os.path.isdir(os.path.join(SCIGEN_DFT_DIR, d))]
    sc_counts = Counter(d.split('_')[0] for d in all_dirs)
    print(f'SCIGEN DFT structures available: {len(all_dirs)}')
    print('\nBy structural constraint:')
    for sc, count in sc_counts.most_common():
        print(f'  {sc:<6s} {count:>5d}')

In [ ]:
# Sample 5 structures per constraint type for speed (~55 total)
import random
from collections import defaultdict
from pymatgen.io.vasp import Poscar

random.seed(42)

# Group directories by constraint type
dirs_by_type = defaultdict(list)
for d in all_dirs:
    sc_type = d.split('_')[0]
    dirs_by_type[sc_type].append(d)

# Sample and parse structures
N_PER_TYPE = 5
scigen_structs = {}  # {dirname: (structure, sc_type)}
failed = 0

for sc_type, dirs in sorted(dirs_by_type.items()):
    sampled = random.sample(dirs, min(N_PER_TYPE, len(dirs)))
    for dirname in sampled:
        contcar = os.path.join(SCIGEN_DFT_DIR, dirname, 'CONTCAR')
        poscar = os.path.join(SCIGEN_DFT_DIR, dirname, 'POSCAR')
        fpath = contcar if os.path.exists(contcar) else poscar
        try:
            s = Poscar.from_file(fpath).structure
            scigen_structs[dirname] = (s, sc_type)
        except Exception:
            failed += 1

print(f'Loaded {len(scigen_structs)} structures ({failed} failed)')
print(f'Constraint types: {sorted(dirs_by_type.keys())}')
print(f'\nSample:')
for name, (s, sc) in list(scigen_structs.items())[:5]:
    print(f'  {name}: {s.composition.reduced_formula} ({s.num_sites} atoms, {sc})')

### Energy prediction across constraint types

Let's predict the energy per atom for each sampled structure using CHGNet.
Structures with high predicted energy relative to elemental references
are likely to be thermodynamically unstable.

In [ ]:
import time
import pandas as pd
import numpy as np

# Compute elemental reference energies if not already available
try:
    elem_refs
except NameError:
    from pymatgen.core import Structure, Lattice, Element
    elem_refs = {}
    for el in ['Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ti', 'V', 'Cr',
               'Li', 'Na', 'Mg', 'Al', 'Si', 'O', 'S', 'N', 'F', 'Cl']:
        try:
            ref_lat = Lattice.cubic(2.87)
            ref_struct = Structure(ref_lat, [el, el], [[0,0,0], [0.5,0.5,0.5]])
            pred = chgnet.predict_structure(ref_struct)
            elem_refs[el] = float(pred['e'])
        except Exception:
            pass

# Predict energy/forces for all sampled structures
results = []
t0 = time.time()

for name, (struct, sc_type) in scigen_structs.items():
    try:
        pred = chgnet.predict_structure(struct)
        e_per_atom = float(pred['e'])
        max_force = float(np.max(np.linalg.norm(pred['f'], axis=1)))

        # Approximate formation energy
        comp = struct.composition
        e_ref_sum = sum(comp[Element(el)] * elem_refs.get(el, 0)
                        for el in comp.get_el_amt_dict())
        e_form = (e_per_atom * struct.num_sites - e_ref_sum) / struct.num_sites

        results.append({
            'name': name,
            'sc_type': sc_type,
            'formula': struct.composition.reduced_formula,
            'n_atoms': struct.num_sites,
            'e_per_atom': e_per_atom,
            'e_form': e_form,
            'max_force': max_force,
            'volume_per_atom': struct.volume / struct.num_sites,
        })
    except Exception as e:
        pass

elapsed = time.time() - t0
results_df = pd.DataFrame(results)

print(f'Predicted {len(results)} structures in {elapsed:.1f}s ({elapsed/len(results)*1000:.0f} ms/structure)')
print(f'\nSummary by constraint type:')
summary = results_df.groupby('sc_type').agg(
    count=('e_per_atom', 'count'),
    mean_e=('e_per_atom', 'mean'),
    mean_eform=('e_form', 'mean'),
    mean_fmax=('max_force', 'mean')
).round(3)
print(summary.to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Formation energy by constraint type
sc_types_sorted = results_df.groupby('sc_type')['e_form'].median().sort_values().index
positions = range(len(sc_types_sorted))
data_by_type = [results_df[results_df['sc_type'] == sc]['e_form'].values for sc in sc_types_sorted]

bp = axes[0].boxplot(data_by_type, positions=positions, widths=0.6, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#42A5F5')
    patch.set_alpha(0.7)
axes[0].set_xticks(positions)
axes[0].set_xticklabels(sc_types_sorted, rotation=45, ha='right')
axes[0].axhline(0, color='green', ls='--', alpha=0.5, label='E_form = 0')
axes[0].axhline(-0.5, color='gray', ls=':', alpha=0.3)
axes[0].set_ylabel('Formation energy (eV/atom)')
axes[0].set_title('CHGNet predicted formation energy by constraint type')
axes[0].legend(fontsize=9)

# Right: Max residual force distribution
axes[1].hist(results_df['max_force'], bins=20, color='#EF5350', edgecolor='white', alpha=0.8)
axes[1].axvline(0.05, color='green', ls='--', label='Converged threshold (0.05 eV/A)')
axes[1].set_xlabel('Max |force| (eV/A)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual forces on DFT-relaxed structures')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Left: lower formation energy = more thermodynamically favorable')
print('Right: low residual forces indicate CHGNet agrees with the DFT-relaxed geometry')

### Relaxation analysis: how far are generated structures from equilibrium?

A key question: **how much does a SCIGEN-generated structure change when relaxed by CHGNet?**
Small displacements suggest the generative model already produces near-equilibrium geometries.
We relax a subset of structures and measure:
- **Energy drop** during relaxation (eV/atom)
- **RMSD** of atomic positions before vs. after

In [ ]:
from chgnet.model.dynamics import StructOptimizer

relaxer = StructOptimizer()

# Relax 2 structures per constraint type (~22 total, ~1-3 min)
relax_results = []
n_per_type = 2
sampled_for_relax = []

for sc_type in sorted(dirs_by_type.keys()):
    type_structs = [(n, s, t) for n, (s, t) in scigen_structs.items() if t == sc_type]
    sampled_for_relax.extend(type_structs[:n_per_type])

print(f'Relaxing {len(sampled_for_relax)} structures...')
t0 = time.time()

for name, struct, sc_type in sampled_for_relax:
    try:
        # Energy before relaxation
        pred_before = chgnet.predict_structure(struct)
        e_before = float(pred_before['e'])

        # Relax
        result = relaxer.relax(struct, verbose=False)
        relaxed = result['final_structure']

        # Energy after
        pred_after = chgnet.predict_structure(relaxed)
        e_after = float(pred_after['e'])

        # RMSD of atomic positions
        cart_before = struct.cart_coords
        cart_after = relaxed.cart_coords
        if len(cart_before) == len(cart_after):
            rmsd = np.sqrt(np.mean(np.sum((cart_before - cart_after)**2, axis=1)))
        else:
            rmsd = float('nan')

        relax_results.append({
            'name': name,
            'sc_type': sc_type,
            'formula': struct.composition.reduced_formula,
            'e_before': e_before,
            'e_after': e_after,
            'e_drop': e_before - e_after,
            'rmsd': rmsd,
        })
        print(f'  {name}: dE={e_before - e_after:+.4f} eV/atom, RMSD={rmsd:.3f} A')
    except Exception as e:
        print(f'  {name}: failed ({e})')

elapsed = time.time() - t0
relax_df = pd.DataFrame(relax_results)
print(f'\nRelaxed {len(relax_df)} structures in {elapsed:.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Energy drop by constraint type
sc_order = relax_df.groupby('sc_type')['e_drop'].median().sort_values(ascending=False).index
data_edrop = [relax_df[relax_df['sc_type'] == sc]['e_drop'].values for sc in sc_order]
bp1 = axes[0].boxplot(data_edrop, positions=range(len(sc_order)), widths=0.5, patch_artist=True)
for p in bp1['boxes']:
    p.set_facecolor('#66BB6A')
    p.set_alpha(0.7)
axes[0].set_xticks(range(len(sc_order)))
axes[0].set_xticklabels(sc_order, rotation=45, ha='right')
axes[0].axhline(0.01, color='gray', ls=':', alpha=0.5, label='0.01 eV/atom')
axes[0].set_ylabel('Energy drop (eV/atom)')
axes[0].set_title('Energy change during CHGNet relaxation')
axes[0].legend(fontsize=9)

# Right: RMSD by constraint type
data_rmsd = [relax_df[relax_df['sc_type'] == sc]['rmsd'].values for sc in sc_order]
bp2 = axes[1].boxplot(data_rmsd, positions=range(len(sc_order)), widths=0.5, patch_artist=True)
for p in bp2['boxes']:
    p.set_facecolor('#FFA726')
    p.set_alpha(0.7)
axes[1].set_xticks(range(len(sc_order)))
axes[1].set_xticklabels(sc_order, rotation=45, ha='right')
axes[1].axhline(0.5, color='red', ls='--', alpha=0.5, label='0.5 A threshold')
axes[1].set_ylabel('RMSD (A)')
axes[1].set_title('Atomic displacement during relaxation')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Small energy drops and RMSD values indicate the DFT-relaxed structures')
print('are already near CHGNet energy minima -- MLIP and DFT agree on the geometry.')

### Screening summary

Let's apply simple stability filters to identify the most promising candidates:
- **Low residual forces** (max |F| < 0.1 eV/A on DFT-relaxed structure)
- **Negative formation energy** (thermodynamically favorable)
- **Small relaxation RMSD** (< 0.5 A, structure is near equilibrium)

In [ ]:
# Apply screening filters
force_threshold = 0.1   # eV/A
rmsd_threshold = 0.5    # Angstrom

# Filter 1: force-based (all predicted structures)
low_force = results_df[results_df['max_force'] < force_threshold]
neg_eform = results_df[results_df['e_form'] < 0]
both = results_df[(results_df['max_force'] < force_threshold) & (results_df['e_form'] < 0)]

print('=== Screening results ===')
print(f'Total structures evaluated:        {len(results_df)}')
print(f'Low residual force (< {force_threshold} eV/A):  {len(low_force)} ({100*len(low_force)/len(results_df):.0f}%)')
print(f'Negative formation energy:         {len(neg_eform)} ({100*len(neg_eform)/len(results_df):.0f}%)')
print(f'Pass both filters:                 {len(both)} ({100*len(both)/len(results_df):.0f}%)')

# Pass rate by constraint type
print(f'\nPass rate by constraint type (both filters):')
for sc_type in sorted(results_df['sc_type'].unique()):
    total = len(results_df[results_df['sc_type'] == sc_type])
    passed = len(both[both['sc_type'] == sc_type])
    bar = '#' * int(20 * passed / max(total, 1))
    print(f'  {sc_type:<6s} {passed:2d}/{total:2d} ({100*passed/max(total,1):5.1f}%) {bar}')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
types = sorted(results_df['sc_type'].unique())
pass_rates = []
for sc in types:
    total = len(results_df[results_df['sc_type'] == sc])
    passed = len(both[both['sc_type'] == sc])
    pass_rates.append(100 * passed / max(total, 1))

colors = ['#4CAF50' if r > 50 else '#FFC107' if r > 20 else '#EF5350' for r in pass_rates]
ax.bar(types, pass_rates, color=colors, edgecolor='white')
ax.set_ylabel('Pass rate (%)')
ax.set_xlabel('Structural constraint')
ax.set_title('MLIP screening pass rate by constraint type')
ax.set_ylim(0, 105)
for i, (sc, r) in enumerate(zip(types, pass_rates)):
    ax.text(i, r + 2, f'{r:.0f}%', ha='center', fontsize=9)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 5.9 The full discovery pipeline

We have now demonstrated every stage of the SCIGEN materials discovery pipeline:

| Stage | Tool | Notebook | Throughput |
|-------|------|----------|------------|
| **1. Generate** candidates | SCIGEN diffusion model | NB 04 | ~1,000 / hour |
| **2. Pre-screen** | Composition + bond filters | NB 04 | instant |
| **3. MLIP screen** | **CHGNet** (this notebook) | **NB 05** | **~1,000 / hour** |
| **4. DFT validation** | VASP | (offline) | ~10 / hour |

In the published work:
- **10 million** structures were generated
- **~100,000** passed pre-screening (composition validity, bond lengths)
- **~25,000** passed MLIP + DFT validation
- **24,743** are the final DFT-confirmed structures we just analyzed

> **MLIP screening reduced the DFT workload by ~400x**, making the entire pipeline feasible.
> Without it, validating 100K structures with DFT would take years of compute time.

This demonstrates why MLIPs like CHGNet are not just a convenience -- they are an
**essential component** of any high-throughput computational materials discovery pipeline.

---
## Key takeaways

1. **Generation without evaluation is useless.** Fast, reliable screening is essential.
2. **MLIPs (like CHGNet) are ~1000x faster than DFT** and accurate enough for initial screening.
3. **Structure relaxation** verifies that generated structures are near energy minima.
4. **Thermodynamic stability** (energy above hull) identifies synthesizable candidates.
5. **SCIGEN-generated materials can be rapidly screened** -- we evaluated ~55 structures in seconds, identified stable candidates across constraint types.
6. **The full pipeline** (generate -> pre-screen -> MLIP screen -> DFT validate) reduced 10M candidates to 24,743 confirmed materials.

---
## What's next?

You have completed the full tutorial series!

**Recap of the pipeline:**
1. **Notebook 01:** Crystal structures, materials data, graph representation, invariances
2. **Notebook 02:** Generative AI concepts -- DDPM on MNIST, connection to materials
3. **Notebook 03:** DiffCSP -- crystal structure diffusion fundamentals
4. **Notebook 04:** SCIGEN -- constrained generation (kagome, honeycomb, etc.)
5. **Notebook 05:** MLIP evaluation -- CHGNet screening of generated materials (this notebook)

For more details, see the SCIGEN paper: [Nature Materials (2025)](https://www.nature.com/articles/s41563-025-02114-9)